# Globus → IRI Token Bootstrap Notebook

<div style="color: red; font-size: 28px; font-weight: bold; text-align: center; border: 3px solid red; padding: 12px; border-radius: 8px;">
⚠️ THIS IS TEMPORARY FOR DEMO AND WILL NOT BE SUPPORTED IN THE FUTURE  
Please use **facility-specific authentication** instead.
</div>

This notebook performs an interactive OAuth2 flow using **Globus Auth**.

It will:
1. Start an OAuth authorization flow
2. Exchange the authorization code for an access token
3. Display expiration details
4. Save the "iri_api"-scoped token to `~/.iri_token.json` as `IRI_API_TOKEN`

## Required Environment Variables
- `GLOBUS_ID`
- `GLOBUS_SECRET`

These may be set in your shell or a `.env` file. To receive GLOBUS_ID and GLOBUS_SECRET, proceed to developers.globus.org and register new Confidential Client app. For more information, please look at Globus documentation.


## 1) Setup and Imports

In [ ]:
import sys
!{sys.executable} -m pip install globus-sdk python-dotenv

In [ ]:
import os
import datetime
import time
import json
import pprint
from pathlib import Path

import globus_sdk
from dotenv import load_dotenv


## 2) Configuration

In [ ]:
# Load environment variables from .env (if present)
load_dotenv()

GLOBUS_ID = os.getenv("GLOBUS_ID")
GLOBUS_SECRET = os.getenv("GLOBUS_SECRET")

# the iri_api resource server details
GLOBUS_RS_ID = os.getenv("GLOBUS_RS_ID", "ed3e577d-f7f3-4639-b96e-ff5a8445d699")
GLOBUS_RS_SCOPE_SUFFIX = os.getenv("GLOBUS_RS_SCOPE_SUFFIX", "iri_api")
GLOBUS_RS_SCOPE = os.getenv("GLOBUS_RS_SCOPE", f"https://auth.globus.org/scopes/{GLOBUS_RS_ID}/{GLOBUS_RS_SCOPE_SUFFIX}")

if not GLOBUS_ID or not GLOBUS_SECRET:
    raise RuntimeError("GLOBUS_ID and/or GLOBUS_SECRET not set in environment.")

print("Using Globus ID:", GLOBUS_ID)


## 3) Start OAuth Flow

In [ ]:
# Create a confidential client
client = globus_sdk.ConfidentialAppAuthClient(GLOBUS_ID, GLOBUS_SECRET)

# Start OAuth flow
client.oauth2_start_flow(
    redirect_uri="http://localhost:5000/callback",  # must match registered redirect URI
    requested_scopes=[
        "openid",
        "profile",
        "email",
        "urn:globus:auth:scope:auth.globus.org:view_identities",
        GLOBUS_RS_SCOPE,
    ],
)

# Get authorization URL
authorize_url = client.oauth2_get_authorize_url(query_params={"prompt": "login"})
print("\nVisit this URL in your browser and authorize:\n")
print(authorize_url)


## 4) Exchange Code for Token

In [ ]:
# After authorizing in the browser, paste the 'code' parameter here
auth_code = input("\nPaste the 'code' parameter from the redirect URL: ").strip()

# Exchange the code for tokens
token_response = client.oauth2_exchange_code_for_tokens(auth_code)

iri_token = None
if GLOBUS_RS_ID in token_response.by_resource_server:
    iri_token_data = token_response.by_resource_server[GLOBUS_RS_ID]
    iri_token = iri_token_data['access_token']
    expires_at = iri_token_data['expires_at_seconds']

    # Convert to human-readable time
    expiration_time = datetime.datetime.fromtimestamp(expires_at)
    print(f"\nIRI API token expires at: {expiration_time}")

    # Calculate how long until expiration
    seconds_until_expiration = expires_at - time.time()
    hours_until_expiration = seconds_until_expiration / 3600
    print(f"Token expires in {hours_until_expiration:.2f} hours")

    print(f"\n=== USE THIS IRI API TOKEN ===")
    print(f"\n{iri_token}")
    print("\n=== TOKEN INTROSPECT ===")
    introspect = client.oauth2_token_introspect(iri_token, include="identity_set_detail,session_info")
    print(introspect)
else:
    print(f"\nERROR: No IRI API token found. Make sure you requested the correct scope.")

## 5) Save Token for Other Notebooks

In [ ]:
home_path = Path("~/.iri_token.json").expanduser()

home_path.parent.mkdir(parents=True, exist_ok=True)

with open(home_path, "w") as f:
    json.dump({"IRI_API_TOKEN": iri_token}, f)
print(f"Token saved: {home_path}")